In [1]:
import pandas as pd

In [15]:
df_siga = pd.read_csv("../data/cadastro/siga-empreendimentos-geracao.csv", sep=";")
df_indiv = df_siga.loc[df_siga['NomEmpreendimento'].str.contains("Lagoa 1|Lagoa 2|Lagoa 3|Lagoa 4")]

In [13]:
df_siga.loc[df_siga['NomEmpreendimento'].str.contains("Lagoa 1|Lagoa 2|Lagoa 3|Lagoa 4")]

,DatGeracaoConjuntoDados,NomEmpreendimento,IdeNucleoCEG,CodCEG,SigUFPrincipal,SigTipoGeracao,DscFaseUsina,DscOrigemCombustivel,DscFonteCombustivel,DscTipoOutorga,...,MdaPotenciaFiscalizadaKw,MdaGarantiaFisicaKw,IdcGeracaoQualificada,NumCoordNEmpreendimento,NumCoordEEmpreendimento,DatInicioVigencia,DatFimVigencia,DscPropriRegimePariticipacao,DscSubBacia,DscMuninicpios
4430,2026-09-01,Lagoa 1,33664,EOL.CV.PB.033664-5.1,PB,EOL,Operação,Eólica,Cinética do vento,Autorização,...,31500,"18700,00",Não,"-6,87904067","-36,81843069",2015-08-04,2050-08-04,100% para LAGOA 1 ENERGIA RENOVAVEL S.A. - 21....,NaN,Santa Luzia - PB
4431,2026-09-01,Lagoa 2,33665,EOL.CV.PB.033665-3.1,PB,EOL,Operação,Eólica,Cinética do vento,Autorização,...,31500,"17500,00",Não,"-6,84091341","-36,79106777",2015-08-04,2050-08-04,100% para LAGOA 2 ENERGIA RENOVAVEL S.A. - 21....,NaN,São José do Sabugi - PB
5606,2026-09-01,Lagoa 3,38029,EOL.CV.PB.038029-6.1,PB,EOL,Operação,Eólica,Cinética do vento,Autorização,...,34650,"17200,00",Não,"-6,87119192","-36,82620039",2018-06-26,2053-06-26,100% para LAGOA 3 ENERGIA RENOVÁVEL S.A - 22.5...,NaN,São José do Sabugi - PB
5607,2026-09-01,Lagoa 4,38030,EOL.CV.PB.038030-0.1,PB,EOL,Operação,Eólica,Cinética do vento,Autorização,...,20790,"10200,00",Não,"-6,88236186","-36,84149027",2018-06-26,2053-06-26,100% para LAGOA 4 ENERGIA RENOVÁVEL S.A. - 22....,NaN,Santa Luzia - PB


In [16]:
# conjunto_usinas = ['Sol do Sertão', 'Futura', 'Assu Sol']
conjunto_usinas = ['Lagoa 1', 'Lagoa 2', 'Lagoa 3', 'Lagoa 4']
df_indiv_coordenadas = df_indiv.assign(
    latitude=pd.to_numeric(
        df_indiv["NumCoordNEmpreendimento"].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    ),
    longitude=pd.to_numeric(
        df_indiv["NumCoordEEmpreendimento"].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    ),
)

df_pontos_medios = (
    pd.concat(
        [
            df_indiv_coordenadas.loc[
                df_indiv_coordenadas["NomEmpreendimento"].str.contains(
                    conjunto, case=False, regex=False, na=False
                )
            ].assign(conjunto=conjunto)
            for conjunto in conjunto_usinas
        ],
        ignore_index=True,
    )
    .dropna(subset=["latitude", "longitude"])
    .groupby("conjunto", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean"),
        numero_usinas=("NomEmpreendimento", "size"),
    )
)

df_pontos_medios.to_parquet("../data/cadastro/df_pontos_medios.parquet")
